## Google Cloud Messaging Essentials

# Google Cloud Pub/Sub: Core Operations and Lifecycle Management

## Introduction

In this lesson, we will explore the core concepts and operations of **Google Cloud Pub/Sub**, a fully managed messaging service that enables reliable, scalable, and asynchronous communication between different parts of your applications.

Cloud Pub/Sub is designed around the concepts of **topics** and **subscriptions**, allowing you to publish messages to a topic and have them delivered to one or more independent subscribers. We will cover how to publish messages, embed custom metadata attributes, configure pull mechanics, and manage message delivery status via acknowledgments.

---

## Publishing Messages to Topics

To broadcast information through Google Cloud Pub/Sub, you publish raw data payloads to a designated topic channel.

### 1. Publishing a Standard Message Payload

```python
from google.cloud import pubsub_v1

# Initialize the network client
publisher = pubsub_v1.PublisherClient()
topic_path = publisher.topic_path('your-project-id', 'your-topic-name')

# Payloads must be transmitted as raw byte strings (b'')
future = publisher.publish(topic_path, b'Hello world!')
print(f'Message published: {future.result()}')

```

**Console Output:**

> Message published: 123456789012345678

### 2. Attaching Custom Meta-Attributes

You can attach arbitrary string-based key-value pairs as attributes alongside your message body. This metadata is highly useful for downstream routing or server-side subscription filtering.

```python
future = publisher.publish(
    topic_path,
    b'Hello with attributes!',
    author='John Doe',
    weeks_on_job='10'
)
print(f'Message published: {future.result()}')

```

**Console Output:**

> Message published: 123456789012345679

---

## Sending Multiple Messages at a Time

When processing high-velocity data streams, publishing items individually introduces unnecessary network round-trips. Cloud Pub/Sub supports high-throughput **batch publishing** to bundle multiple events together.

```python
from google.cloud import pubsub_v1

publisher = pubsub_v1.PublisherClient()
topic_path = publisher.topic_path('your-project-id', 'your-topic-name')

# Staging collection of message payloads and their associated dictionary attributes
messages = [
    (b'This is the content of message 1', {'author': 'Jane Doe'}),
    (b'This is the content of message 2', {'author': 'John Smith'}),
]

futures = []
for data, attrs in messages:
    # Under the hood, the client library intelligently batches these requests
    future = publisher.publish(topic_path, data, **attrs)
    futures.append(future)

# Block until all background thread dispatches successfully resolve
for future in futures:
    print(f'Message published: {future.result()}')

```

**Console Output:**

> Message published: 123456789012345680
> Message published: 123456789012345681

> 💡 **Client-Side Optimization Note:** The Python SDK handles batching automatically using default time thresholds (e.g., maximum delay of 10ms or 100 messages). You can customize these boundaries by passing an explicit BatchSettings object into the PublisherClient constructor to fit your specific throughput requirements.

---

## Message Delivery and Retrieval

Once an item hits a topic, Pub/Sub clones and pushes that record out to every subscription attached to that channel. There are two primary architectural topographies used to ingest these messages:

* **Pull Subscriptions:** Your application acts as the driver. It explicitly makes a synchronous or streaming request to the server to check for, extract, and handle pending payloads.
* **Push Subscriptions:** The Pub/Sub router acts as the driver. It immediately hits a pre-configured, publicly available HTTP/HTTPS webhook target endpoint (such as Cloud Run or an external API gateway) as soon as new data arrives.

### Synchronously Pulling Messages from a Subscription

```python
from google.cloud import pubsub_v1

subscriber = pubsub_v1.SubscriberClient()
subscription_path = subscriber.subscription_path('your-project-id', 'your-subscription-name')

# Explicitly pull up to 10 unacknowledged messages out of the storage buffer
response = subscriber.pull(
    request={
        "subscription": subscription_path,
        "max_messages": 10,
    }
)

for received_message in response.received_messages:
    print(f"Received: {received_message.message.data}")
    print(f"Message ID: {received_message.message.message_id}")
    print(f"Publish time: {received_message.message.publish_time}")
    
    if received_message.message.attributes:
        print(f"Attributes: {dict(received_message.message.attributes)}")
        
    # CRITICAL: Always acknowledge receipt after successful downstream execution
    subscriber.acknowledge(
        request={
            "subscription": subscription_path,
            "ack_ids": [received_message.ack_id],
        }
    )

```

**Console Output:**

> Received: b'Hello world!'
> Message ID: 123456789012345678
> Publish time: 2026-05-22 10:30:45.123456+00:00
> Received: b'Hello with attributes!'
> Message ID: 123456789012345679
> Publish time: 2026-05-22 10:30:45.234567+00:00
> Attributes: {'author': 'John Doe', 'weeks_on_job': '10'}

---

## Acknowledging and Deleting Messages

The complete message delivery life cycle guarantees an **at-least-once** execution contract:

```python
subscriber.acknowledge(
    request={
        "subscription": subscription_path,
        "ack_ids": [received_message.ack_id],
    }
)
print("Message acknowledged successfully")

```

1. **Lease Lock:** When a worker pulls a message, that message is temporarily placed into an invisible "leased" lock state for a set timeframe known as the **Acknowledgment Deadline** (defaulting to 10 seconds).
2. **The Ack Contract:** If your worker processes the message and responds back with an explicit .acknowledge() payload containing the unique ack_id string before the lock expires, Pub/Sub safely cleanses and drops that document record out of the storage buffer queue.
3. **Automatic Fault Recovery:** If your worker crashes, loses internet connectivity, or experiences a deep thread freeze mid-execution, the deadline will lapse. Pub/Sub recognizes this timeout as a processing failure, drops the lock, and automatically places the target item back in the queue for redelivery to a healthy worker thread.

---

## Summary

In this lesson, you mastered how to utilize Google Cloud Pub/Sub to build scalable, asynchronous backend systems. You learned how to instantiate a publication client, combine messages into high-throughput batches, read incoming streams via pull subscriptions, and complete data execution flows using explicit consumer confirmations.

Keep practicing these core infrastructure patterns to build highly decentralized, bulletproof cloud-based applications!

## Run Google Cloud Pub/Sub Messaging Script

In this task, your goal is to run a pre-written Python script that employs Google Cloud Pub/Sub messaging services. The script is designed to create both a standard and an ordered (FIFO-like) topic with subscriptions, then dispatch, receive, and acknowledge various messages within these topics. Make sure to examine the script's structure and commands before executing it for a better understanding of its procedures.

Important Note: Running scripts can modify the resources in our Google Cloud simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, keep in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
from google.cloud import pubsub_v1
import os
import time

# Create publisher and subscriber clients
publisher = pubsub_v1.PublisherClient()
subscriber = pubsub_v1.SubscriberClient()

# Project ID
project_id = os.getenv('GOOGLE_CLOUD_PROJECT', 'your-project-id')

# Create topic and subscription paths
standard_topic_path = publisher.topic_path(project_id, 'test-topic')
fifo_topic_path = publisher.topic_path(project_id, 'test-topic-fifo')
standard_subscription_path = subscriber.subscription_path(project_id, 'test-subscription')
fifo_subscription_path = subscriber.subscription_path(project_id, 'test-subscription-fifo')

# Create standard topic and subscription
try:
    publisher.create_topic(request={"name": standard_topic_path})
except Exception:
    pass  # Topic may already exist

try:
    subscriber.create_subscription(
        request={"name": standard_subscription_path, "topic": standard_topic_path}
    )
except Exception:
    pass  # Subscription may already exist

# Create FIFO-like topic and subscription with message ordering enabled
try:
    publisher.create_topic(request={"name": fifo_topic_path})
except Exception:
    pass  # Topic may already exist

# Ensure FIFO subscription is properly configured
try:
    # Try to delete existing subscription
    subscriber.delete_subscription(request={"subscription": fifo_subscription_path})
    time.sleep(1)  # Brief delay to ensure deletion is processed
except Exception:
    pass  # Subscription may not exist

# Create FIFO subscription with message ordering enabled
try:
    subscriber.create_subscription(
        request={
            "name": fifo_subscription_path, 
            "topic": fifo_topic_path,
            "enable_message_ordering": True
        }
    )
    print("FIFO subscription created with message ordering enabled")
except Exception as e:
    print(f"Error creating FIFO subscription: {e}")

# Simple message
data = "Hello world!".encode("utf-8")
future = publisher.publish(standard_topic_path, data)
print(f"Published message ID: {future.result()}")

# Message with custom attributes
data = "Hello with attributes!".encode("utf-8")
future = publisher.publish(
    standard_topic_path,
    data,
    Author="John Doe",
    WeeksOnJob="10"
)
print(f"Published message with attributes ID: {future.result()}")

# Message with delay simulation (Pub/Sub doesn't have native delay)
# In practice, you would use Cloud Tasks for delayed messaging
data = "Hello with delay!".encode("utf-8")
future = publisher.publish(
    standard_topic_path, 
    data,
    delay_seconds="10"  # This is just an attribute, not functional delay
)
print(f"Published delayed message ID: {future.result()}")

# Message to FIFO-like topic with ordering key
try:
    data = "Hello from FIFO queue!".encode("utf-8")
    future = publisher.publish(
        fifo_topic_path,
        data,
        ordering_key="group1"  # Ensures message ordering
    )
    print(f"Published FIFO message ID: {future.result()}")
except Exception as e:
    print(f"Error publishing FIFO message: {e}")
    # Publish without ordering key as fallback
    future = publisher.publish(
        fifo_topic_path,
        data
    )
    print(f"Published FIFO message without ordering key ID: {future.result()}")

# Receive messages
response = subscriber.pull(
    request={
        "subscription": standard_subscription_path,
        "max_messages": 10
    }
)

if response.received_messages:
    for received_message in response.received_messages:
        message = received_message.message
        print(f"Processing message: {message.data.decode('utf-8')}")

        # Acknowledge (equivalent to delete in SQS)
        subscriber.acknowledge(
            request={
                "subscription": standard_subscription_path,
                "ack_ids": [received_message.ack_id]
            }
        )
        print(f"Acknowledged message: {message.message_id}")
else:
    print("No messages to process!")

```

## Add StartingDate Attribute to Google Cloud Pub/Sub Message

In this task, you will modify an existing Python script that interacts with Google Cloud Pub/Sub. The current script creates a Pub/Sub topic and subscription, and publishes a message with two attributes: Author and WeeksOnJob. Your task is to add a third attribute, StartingDate, representing the job's starting date in a YYYY-MM-DD format. After implementing your changes, run the script to publish the modified message to the topic.

Important Note: Executing scripts can alter the resources in our GCP environment. To return to the original state, you may use the reset button located in the top right corner. Nonetheless, remember that resetting will eliminate any code modifications. To maintain your code through a reset, consider copying it to the clipboard.

```python
from google.cloud import pubsub_v1
import os

# Create publisher and subscriber clients
publisher = pubsub_v1.PublisherClient()
subscriber = pubsub_v1.SubscriberClient()

# Project ID
project_id = os.getenv('GOOGLE_CLOUD_PROJECT', 'your-project-id')

# Create topic and subscription paths
topic_path = publisher.topic_path(project_id, 'test-topic')
subscription_path = subscriber.subscription_path(project_id, 'test-subscription')

# Create topic
try:
    publisher.create_topic(request={"name": topic_path})
    print(f"Topic created: {topic_path}")
except Exception:
    print(f"Topic already exists: {topic_path}")

# Create subscription
try:
    subscriber.create_subscription(
        request={"name": subscription_path, "topic": topic_path}
    )
    print(f"Subscription created: {subscription_path}")
except Exception:
    print(f"Subscription already exists: {subscription_path}")

# Message with custom attributes
data = "Hello with attributes!".encode("utf-8")
future = publisher.publish(
    topic_path,
    data,
    Author="John Doe",
    WeeksOnJob="10"
    # TODO: Add a new attribute 'StartingDate' in 'YYYY-MM-DD' format
)
print(f"Message published with ID: {future.result()}")
```
Here is the completed script with the StartingDate attribute added to the publishing request in the required YYYY-MM-DD format.

```python
from google.cloud import pubsub_v1
import os

# Create publisher and subscriber clients
publisher = pubsub_v1.PublisherClient()
subscriber = pubsub_v1.SubscriberClient()

# Project ID
project_id = os.getenv('GOOGLE_CLOUD_PROJECT', 'your-project-id')

# Create topic and subscription paths
topic_path = publisher.topic_path(project_id, 'test-topic')
subscription_path = subscriber.subscription_path(project_id, 'test-subscription')

# Create topic
try:
    publisher.create_topic(request={"name": topic_path})
    print(f"Topic created: {topic_path}")
except Exception:
    print(f"Topic already exists: {topic_path}")

# Create subscription
try:
    subscriber.create_subscription(
        request={"name": subscription_path, "topic": topic_path}
    )
    print(f"Subscription created: {subscription_path}")
except Exception:
    print(f"Subscription already exists: {subscription_path}")

# Message with custom attributes
data = "Hello with attributes!".encode("utf-8")

# --- TODO FIXED: Added the 'StartingDate' attribute in YYYY-MM-DD format ---
future = publisher.publish(
    topic_path,
    data,
    Author="John Doe",
    WeeksOnJob="10",
    StartingDate="2026-05-23"  
)
print(f"Message published with ID: {future.result()}")

```

---

### Mechanics Behind Custom Attributes

When you append arbitrary arguments like StartingDate="2026-05-23" to the .publish() method, the Python SDK automatically parses them into a dictionary of string-to-string mappings. Under the hood, this metadata is bundled into the attributes field of the underlying PubsubMessage object.

Downstream consumer applications can read this dictionary out of the message wrapper *without* needing to unpack or deserialize the main binary message payload data (data). This is incredibly powerful for lightweight routing engines, log analytics, or using **Filter Expressions** on your subscription layers to route specific dates or authors to distinct backend worker pods.

## Complete Google Cloud Pub/Sub Message Pull and Acknowledgment Script

In this task, you are given a script that interacts with an existing Google Cloud Pub/Sub subscription called 'test-subscription'. This subscription contains motivational messages. Your task is to complete the script by adding functionality to pull a message from the subscription, print the content of the message, and then acknowledge it to remove it from the subscription. Good luck!

Important Note: Running scripts can modify the resources in our Google Cloud simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, keep in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
from google.cloud import pubsub_v1
import os

# Create publisher and subscriber clients
publisher = pubsub_v1.PublisherClient()
subscriber = pubsub_v1.SubscriberClient()

# Project ID
project_id = os.getenv('GOOGLE_CLOUD_PROJECT', 'your-project-id')

# Create topic and subscription paths
topic_path = publisher.topic_path(project_id, 'test-topic')
subscription_path = subscriber.subscription_path(project_id, 'test-subscription')

# Create topic
try:
    publisher.create_topic(request={"name": topic_path})
    print(f"Topic created: {topic_path}")
except Exception:
    print(f"Topic already exists: {topic_path}")

# Create subscription
try:
    subscriber.create_subscription(
        request={"name": subscription_path, "topic": topic_path}
    )
    print(f"Subscription created: {subscription_path}")
except Exception:
    print(f"Subscription already exists: {subscription_path}")

# Message with modified attributes
data = "Hello with modified attributes!".encode("utf-8")
future = publisher.publish(
    topic_path,
    data,
    Author="John Doe",
    StartingDate="2021-01-01",
    WeeksOnJob="10"
)
print(f"Message published with ID: {future.result()}")

# Pull a message from the subscription

# Print the received message

# Acknowledge the received message to remove it from the subscription

```

Here is the completed script containing the added execution logic to cleanly pull the pending message from your subscription, decode its binary payload, and acknowledge its receipt back to the GCP Pub/Sub server engine.

```python
from google.cloud import pubsub_v1
import os

# Create publisher and subscriber clients
publisher = pubsub_v1.PublisherClient()
subscriber = pubsub_v1.SubscriberClient()

# Project ID
project_id = os.getenv('GOOGLE_CLOUD_PROJECT', 'your-project-id')

# Create topic and subscription paths
topic_path = publisher.topic_path(project_id, 'test-topic')
subscription_path = subscriber.subscription_path(project_id, 'test-subscription')

# Create topic
try:
    publisher.create_topic(request={"name": topic_path})
    print(f"Topic created: {topic_path}")
except Exception:
    print(f"Topic already exists: {topic_path}")

# Create subscription
try:
    subscriber.create_subscription(
        request={"name": subscription_path, "topic": topic_path}
    )
    print(f"Subscription created: {subscription_path}")
except Exception:
    print(f"Subscription already exists: {subscription_path}")

# Message with modified attributes
data = "Hello with modified attributes!".encode("utf-8")
future = publisher.publish(
    topic_path,
    data,
    Author="John Doe",
    StartingDate="2021-01-01",
    WeeksOnJob="10"
)
print(f"Message published with ID: {future.result()}\n")

# ========================================================
# TODO FIXED: PULL, PRINT, AND ACKNOWLEDGE THE MESSAGE
# ========================================================

# 1. Pull a message from the subscription synchronously
response = subscriber.pull(
    request={
        "subscription": subscription_path,
        "max_messages": 1  # We only need to fetch the single message we just created
    }
)

# Verify if any messages were returned in the pull response window
if response.received_messages:
    received_message = response.received_messages[0]
    message_payload = received_message.message
    
    # 2. Print the received message details and decode the byte string payload
    print("--- Message Received ---")
    print(f"Payload Body: {message_payload.data.decode('utf-8')}")
    print(f"Message ID: {message_payload.message_id}")
    print(f"Metadata Attributes: {dict(message_payload.attributes)}")

    # 3. Acknowledge the received message to remove it permanently from the queue buffer
    subscriber.acknowledge(
        request={
            "subscription": subscription_path,
            "ack_ids": [received_message.ack_id]
        }
    )
    print("\nStatus: Message acknowledged successfully and dropped from the queue.")
else:
    print("Status: No message found in the subscription buffer queue.")

```

---

### Understanding the Execution Flow

1. **`subscriber.pull(...)`**: This initiates a synchronous pull request. The subscription holds the message and marks it as temporarily invisible to any other consumer threads for the duration of its defined acknowledgment deadline.
2. **`.data.decode('utf-8')`**: Because data in transit across GCP's high-performance gRPC network is strictly encoded as raw binary byte structures, we apply Python's string conversion mapping to make the payload human-readable.
3. **`subscriber.acknowledge(...)`**: This step is critical. If your application logic fails to send this confirmation payload containing the unique tracking `ack_id` string before the deadline timer hits zero, Pub/Sub assumes your consumer process crashed and will put the message back in line for a redelivery attempt. Sending this acknowledgment guarantees the message lifecycle is finalized cleanly.

## Send Multiple Messages to Google Cloud Pub/Sub with Message Group ID

Good work on your progress! In this task, you'll send multiple messages to a Google Cloud Pub/Sub topic. A topic is created initially for you. You're tasked to send two messages to this topic, with each message having the same message_group_id attribute of 'CodeSignal2021'. The content of these messages doesn't matter. This task aims to help you learn how to properly send multiple messages to a Pub/Sub topic using Python (google-cloud-pubsub).

Important Note: Running scripts can modify the resources in our GCP simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, keep in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
from google.cloud import pubsub_v1
import os

# Create publisher client
publisher = pubsub_v1.PublisherClient()

# Project ID  
project_id = os.getenv('GOOGLE_CLOUD_PROJECT', 'test-project')

# Create topic path
topic_path = publisher.topic_path(project_id, 'student-topic')

# Create topic
try:
    publisher.create_topic(request={"name": topic_path})
except Exception:
    pass  # Topic already exists

# TODO: Send 2 messages to the topic with the same message_group_id of 'CodeSignal2021'.

```

Here is the completed script containing the added execution logic to publish two distinct messages to your topic, with each payload explicitly carrying the custom `message_group_id` attribute set to `'CodeSignal2021'`.

```python
from google.cloud import pubsub_v1
import os

# Create publisher client
publisher = pubsub_v1.PublisherClient()

# Project ID  
project_id = os.getenv('GOOGLE_CLOUD_PROJECT', 'test-project')

# Create topic path
topic_path = publisher.topic_path(project_id, 'student-topic')

# Create topic
try:
    publisher.create_topic(request={"name": topic_path})
except Exception:
    pass  # Topic already exists

# ========================================================
# TODO FIXED: SEND 2 MESSAGES WITH CUSTOM METADATA ATTRS
# ========================================================

# Content of the messages (can be any string, converted to raw bytes)
message_contents = [
    "First grouped transaction stream payload",
    "Second grouped transaction stream payload"
]

# Track your asynchronous futures
futures = []

for text in message_contents:
    # Encode string into bytes for cloud transmission
    data = text.encode("utf-8")
    
    # Publish the payload along with the specific metadata attribute
    future = publisher.publish(
        topic_path,
        data,
        message_group_id="CodeSignal2021"
    )
    futures.append(future)

# Block and print results to verify successful ingestion
for index, future in enumerate(futures, start=1):
    print(f"Message {index} successfully published with ID: {future.result()}")

```

---

### Key Architectural Takeaway

When you append keyword arguments like `message_group_id="CodeSignal2021"` directly inside the `.publish()` method, the underlying Python SDK extracts those variables and wraps them cleanly into the message's custom metadata dictionary (`attributes`).

This setup allows downstream consumers or server-side **Filter Expressions** to scan, sort, or quarantine data belonging to a particular grouping index without spending compute resources unpacking the primary message binary body payload.

## Reading and Acknowledging Pub/Sub Messages

Great progress! This task is designed to reinforce your understanding of reading and acknowledging messages from a Google Cloud Pub/Sub subscription. You'll be implementing Python script which reads and acknowledges messages from the Pub/Sub subscription.

Initially, a topic named 'student-topic' and a subscription named 'student-subscription' are created, and 2 messages are sent to this topic. Your task is to:

    Receive both messages from the subscription in a single call using synchronous pull. Print their contents.
    Acknowledge both messages from the subscription.

The messages should be processed in the order they were added (First-In-First-Out).

Important Note: Running scripts can modify the resources in our GCP simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, keep in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
from google.cloud import pubsub_v1
import os

# Create publisher and subscriber clients
publisher = pubsub_v1.PublisherClient()
subscriber = pubsub_v1.SubscriberClient()

# Project ID
project_id = os.getenv('GOOGLE_CLOUD_PROJECT', 'test-project')

# Create topic and subscription paths
topic_path = publisher.topic_path(project_id, 'student-topic')
subscription_path = subscriber.subscription_path(project_id, 'student-subscription')

# Create topic
try:
    publisher.create_topic(request={"name": topic_path})
except Exception:
    pass  # Topic already exists

# Create subscription
try:
    subscriber.create_subscription(
        request={"name": subscription_path, "topic": topic_path}
    )
except Exception:
    pass  # Subscription already exists

# Send 2 messages to the topic
future1 = publisher.publish(topic_path, b'First Message')
future2 = publisher.publish(topic_path, b'Second Message')

# Wait for messages to be published
future1.result()
future2.result()

# TODO: Receive both the messages from the subscription in a single call using synchronous pull. Print their contents.
# TODO: Acknowledge both messages from the subscription.

```

Here is the completed script implementing the synchronous pull to fetch both messages in a single call, decode their raw binary payloads, and acknowledge them simultaneously using their unique delivery keys.

```python
from google.cloud import pubsub_v1
import os

# Create publisher and subscriber clients
publisher = pubsub_v1.PublisherClient()
subscriber = pubsub_v1.SubscriberClient()

# Project ID
project_id = os.getenv('GOOGLE_CLOUD_PROJECT', 'test-project')

# Create topic and subscription paths
topic_path = publisher.topic_path(project_id, 'student-topic')
subscription_path = subscriber.subscription_path(project_id, 'student-subscription')

# Create topic
try:
    publisher.create_topic(request={"name": topic_path})
except Exception:
    pass  # Topic already exists

# Create subscription
try:
    subscriber.create_subscription(
        request={"name": subscription_path, "topic": topic_path}
    )
except Exception:
    pass  # Subscription already exists

# Send 2 messages to the topic
future1 = publisher.publish(topic_path, b'First Message')
future2 = publisher.publish(topic_path, b'Second Message')

# Wait for messages to be published
future1.result()
future2.result()

# ========================================================
# TODO FIXED: PULL AND ACKNOWLEDGE BOTH MESSAGES
# ========================================================

# 1. Pull both messages from the subscription in a single synchronous request
response = subscriber.pull(
    request={
        "subscription": subscription_path,
        "max_messages": 2  # Set to pull up to 2 items simultaneously
    }
)

# Initialize a list to collect tracking IDs for bulk acknowledgment
ack_ids = []

if response.received_messages:
    print(f"Successfully retrieved {len(response.received_messages)} message(s):\n")
    
    for received_message in response.received_messages:
        # Decode the binary byte string payload into standard text
        message_text = received_message.message.data.decode('utf-8')
        print(f"Processing Content: '{message_text}' (ID: {received_message.message.message_id})")
        
        # Collect the target confirmation token
        ack_ids.append(received_message.ack_id)

    # 2. Acknowledge both messages in a single atomic bulk network transaction request
    subscriber.acknowledge(
        request={
            "subscription": subscription_path,
            "ack_ids": ack_ids
        }
    )
    print("\nStatus: Both messages acknowledged in bulk and cleared from the queue.")
else:
    print("Status: No messages were found in the storage buffer queue.")

```

---

### Architectural Insight: Batch Processing over individual Network Trips

Notice how `ack_ids` is managed as a Python list engine. Rather than running a resource-heavy network roundtrip call inside the loop for every single message, the optimal pattern is to collect all `ack_id` strings inside your thread processing chunk and hand them over to `subscriber.acknowledge()` in **one bulk batch call**. This minimizes gRPC request headers, reduces runtime latency, and protects your consumer microservices from hitches under heavy ingestion loops.